# Assignment 1 - Person 1 part: Similarity Metrics (B), Evaluation (E), Analysis (F16-18), Bonus (top-10)

Dataset: F. Maxwell Harper and Joseph A. Konstan, "The MovieLens Datasets: History and Context," ACM TiiS, 2015.

**AI disclosure:** every cell/section between `AI-ASSISTED START` and `AI-ASSISTED END` markers was generated with Claude (Anthropic).
Sections marked `TODO (own words)` must be rewritten/filled by me.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ratings = pd.read_csv("data/ratings.csv")
movies  = pd.read_csv("data/movies.csv")
matrix = ratings.pivot_table(index="userId", columns="movieId", values="rating")
print(matrix.shape)

## Part B - Similarity metrics from scratch

### Hand calculation (worked example, Section 3.2)
<!-- ===== AI-ASSISTED START (Claude): hand calculation ===== -->
A = [5, 4, 5, 3], B = [3, 2, 3, 1]  (B = A - 2 on every movie)

**Euclidean.** differences = [2, 2, 2, 2]; squares = [4, 4, 4, 4]; sum = 16; d = sqrt(16) = 4.
Similarity = 1 / (1 + 4) = **0.2**. (The assignment's "4" is the *distance*.)

**Cosine.** A.B = 5*3 + 4*2 + 5*3 + 3*1 = 15 + 8 + 15 + 3 = 41.
|A|^2 = 25 + 16 + 25 + 9 = 75; |B|^2 = 9 + 4 + 9 + 1 = 23.
cos = 41 / sqrt(75 * 23) = 41 / sqrt(1725) = 41 / 41.533 = **0.9872**.

**Pearson.** mean(A) = 17/4 = 4.25; mean(B) = 9/4 = 2.25.
A - mean = [0.75, -0.25, 0.75, -1.25]; B - mean = [0.75, -0.25, 0.75, -1.25]. The two centered vectors are **identical**, so cosine of identical vectors = **1.0**.

**Why Pearson = 1.0 but cosine = 0.987:** B is A shifted by a constant (-2). Cosine is invariant to *multiplying* a vector by a positive constant (scale), but a constant *added* to every entry changes the angle between the vectors. Pearson subtracts each user's own mean first, so the additive offset (generous vs harsh baseline) disappears and only the pattern of likes/dislikes remains.
<!-- ===== AI-ASSISTED END ===== -->

In [ ]:
def get_co_rated(matrix, a, b, axis="index"):
    if axis == "index":
        va, vb = matrix.loc[a], matrix.loc[b] # hadelo 2 rows
    elif axis == "columns":
        va, vb = matrix[a], matrix[b] # hadeelo 2 cols
    else:
        raise ValueError("axis must be 'index' or 'columns'") #wrong choice nigga
    mask = va.notna() & vb.notna() # keep only entries BOTH rated - NaN is never turned into 0
    return va[mask].to_numpy(dtype=float), vb[mask].to_numpy(dtype=float)

def euclidean_distance(vec1, vec2):
    return float(np.sqrt(np.sum((vec1 - vec2) ** 2)))

def euclidean_similarity(vec1, vec2):
    return 1.0 / (1.0 + euclidean_distance(vec1, vec2))

def cosine_similarity(vec1, vec2):
    denom = np.linalg.norm(vec1) * np.linalg.norm(vec2)
    if denom == 0: # zero vector -> angle undefined, treat as no similarity
        return 0.0
    return float(np.dot(vec1, vec2) / denom)

def pearson_similarity(vec1, vec2):
    # mean-center each vector, then take cosine
    return cosine_similarity(vec1 - vec1.mean(), vec2 - vec2.mean())

In [ ]:
# Sanity check against Section 3.2 (own test code)
A = np.array([5, 4, 5, 3.]); B = np.array([3, 2, 3, 1.])
print("euclidean distance  :", euclidean_distance(A, B))     # 4.0
print("euclidean similarity:", euclidean_similarity(A, B))   # 0.2
print("cosine              :", round(cosine_similarity(A, B), 3))   # 0.987
print("pearson             :", pearson_similarity(A, B))     # 1.0

# get_co_rated on a tiny hand-made matrix with NaNs
toy = pd.DataFrame({10: [5, 4, np.nan], 20: [3, np.nan, 2], 30: [4, 5, 1], 40: [np.nan, 2, 3]},
                   index=["u1", "u2", "u3"])
print(toy)
print(get_co_rated(toy, "u1", "u2", "index")) # only item 10 and 30 -> ([5,4],[4,5])
print(get_co_rated(toy, 10, 30, "columns")) # only users u1,u2,... where both rated

## Part E - Evaluation

<!-- ===== AI-ASSISTED START (Claude): design notes ===== -->
* The split is done **per user**: for each user, 20% of their ratings go to test (at least 1 stays in train, so nobody is left empty).
* The rating matrix used for predictions must be built from **train only**; otherwise the test ratings leak into the neighbours.
* `evaluate` counts a prediction as *impossible* (`None`) when the predictor returns `None` or the user/item does not exist in the train matrix. RMSE/MAE are computed only on possible predictions, and `frac_none` is reported next to them.
* Caveat: different metrics/k produce different `frac_none`, so RMSE values are not computed on exactly the same rows. Compare them together with `frac_none`.
<!-- ===== AI-ASSISTED END ===== -->

In [ ]:
# ===== AI-ASSISTED START (Claude): train/test split =====
def train_test_split_ratings(ratings_df, test_fraction=0.2, seed=42):
    rng = np.random.default_rng(seed)
    test_idx = []
    for user, grp in ratings_df.groupby("userId"):
        n = len(grp)
        n_test = int(round(n * test_fraction))
        n_test = min(n_test, n - 1)              # always keep >= 1 rating in train
        if n_test <= 0:
            continue
        test_idx.extend(rng.choice(grp.index.to_numpy(), size=n_test, replace=False))
    test_df  = ratings_df.loc[test_idx]
    train_df = ratings_df.drop(index=test_idx)
    return train_df.reset_index(drop=True), test_df.reset_index(drop=True)

train_df, test_df = train_test_split_ratings(ratings, 0.2)
print(len(train_df), len(test_df), len(train_df) + len(test_df) == len(ratings))
# own check: every user still has train ratings, and ~20% of each user's ratings are in test
print((test_df.groupby("userId").size() / ratings.groupby("userId").size()).describe())
print(set(train_df.userId) == set(ratings.userId))

train_matrix = train_df.pivot_table(index="userId", columns="movieId", values="rating")
# ===== AI-ASSISTED END =====

In [ ]:
# ===== AI-ASSISTED START (Claude): STUB predictors =====
# TEMPORARY stand-ins with the agreed signature. At merge time DELETE this cell and use
# Person 2's real predict_rating_user_based / predict_rating_item_based.
def predict_rating_user_based(matrix, target_user, target_item, k, similarity_fn):
    m = matrix.loc[target_user].mean()
    return None if np.isnan(m) else float(m)

def predict_rating_item_based(matrix, target_user, target_item, k, similarity_fn):
    if target_item not in matrix.columns:
        return None
    m = matrix[target_item].mean()
    return None if np.isnan(m) else float(m)
# ===== AI-ASSISTED END =====

In [ ]:
def evaluate(matrix, test_df, k, similarity_fn, method, max_test=None, seed=42):
    #mtnsash en method: "user" or "item". Returns dict(rmse, mae, frac_none, n_pred, n_total).
    predict = {"user": predict_rating_user_based, "item": predict_rating_item_based}[method]
    rows = test_df
    if max_test is not None and len(test_df) > max_test: # optional speed-up: fixed random subset
        rows = test_df.sample(max_test, random_state=seed)
    errors, n_none = [], 0
    for u, i, r in zip(rows["userId"], rows["movieId"], rows["rating"]):
        if u not in matrix.index or i not in matrix.columns:  # unseen user/item in train -> impossible
            n_none += 1
            continue
        p = predict(matrix, u, i, k, similarity_fn)
        if p is None:
            n_none += 1
        else:
            errors.append(p - r)
    errors = np.array(errors)
    n_total = len(rows)
    if len(errors) == 0:
        return {"rmse": np.nan, "mae": np.nan, "frac_none": 1.0, "n_pred": 0, "n_total": n_total}
    return {"rmse": float(np.sqrt(np.mean(errors ** 2))),
            "mae": float(np.mean(np.abs(errors))),
            "frac_none": n_none / n_total,
            "n_pred": len(errors), "n_total": n_total}

def run_grid(matrix, test_df, ks=(5, 10, 20, 50), max_test=None):
    sims = {"euclidean": euclidean_similarity, "cosine": cosine_similarity, "pearson": pearson_similarity}
    out = []
    for method in ["user", "item"]:
        for name, fn in sims.items():
            for k in ks:
                res = evaluate(matrix, test_df, k, fn, method, max_test=max_test)
                out.append({"method": method + "-based", "similarity": name, "k": k, **res})
                print(out[-1])
    return pd.DataFrame(out)

print(evaluate(train_matrix, test_df, 10, pearson_similarity, "user"))
print(evaluate(train_matrix, test_df, 10, pearson_similarity, "item"))

In [ ]:
# ===== AI-ASSISTED START (Claude): run full grid + table =====
# RUN THIS ONLY AFTER MERGING Person 2's real predictors.
# The real predictors are slow (they recompute similarities for every test rating), so start with
# max_test=2000 to get a fast table, then rerun with max_test=None if time allows.
results = run_grid(train_matrix, test_df, max_test=2000)

table = results.pivot_table(index=["method", "similarity"], columns="k", values=["rmse", "mae", "frac_none"])
display(table.round(4))
# ===== AI-ASSISTED END =====

## Part F - Analysis

### Q16 - Which metric performed best? (fill after the real grid is run)
**TODO: copy numbers from your own table.** Template (replace `<...>`, delete what doesn't hold):

Best configuration in my table: `<method>` with `<metric>` and k = `<k>`, RMSE = `<..>`, MAE = `<..>`, `<..>`% of test ratings had no prediction.
Worst: `<...>`.

*Does it match Section 3.2?* Section 3.2 predicts that Pearson (mean-centering) should beat cosine, and cosine should beat Euclidean, because Euclidean confuses "different baseline" with "different taste". `<Yes / no>`: in my results, `<metric A>` beat `<metric B>` by `<difference>` RMSE.

*Where it may disagree (check each against your table, keep only what you observe):*
* With co-rated-only data, any two users who share just 2 items have Pearson = +1 or -1 exactly, so Pearson is noisy for low-overlap pairs and can pick bad neighbours at small k.
* Cosine on all-positive ratings is always high (most values 0.9+), so it barely separates good neighbours from bad ones; it only measures angle, not how many items overlap.
* Euclidean similarity 1/(1+d) rewards pairs with *few* co-rated items (a small sum gives a small d), so it can rank users with tiny overlap as very similar, which can hurt it. But because raw predictions are weighted averages of raw ratings (no mean-centering), the bias issue is smaller than the theory suggests, so Euclidean can still be competitive.
* Larger k usually helps until noisy neighbours are included; `<describe what your k trend shows>`.
* User-based vs item-based: `<which was better and by how much; which had a larger frac_none>`.

### Q17 - My own example where Euclidean and Pearson rank the same pairs differently
<!-- ===== AI-ASSISTED START (Claude): Q17 example ===== -->
Four movies, three users:

* A = [2, 3, 4, 5]  (target user: rates later movies higher)
* B = [1, 2, 3, 4]  (same pattern as A, harsher by 1 point)
* C = [3, 3, 4, 4]  (flatter, less sure about the pattern)

**Pair (A, B).**
Euclidean: differences = [1, 1, 1, 1], d = sqrt(4) = 2, similarity = 1/(1+2) = **0.333**.
Pearson: mean(A) = 3.5, mean(B) = 2.5; both centered vectors are [-1.5, -0.5, 0.5, 1.5], identical, so Pearson = **1.0**.

**Pair (A, C).**
Euclidean: differences = [1, 0, 0, 1], d = sqrt(2) = 1.414, similarity = 1/(1+1.414) = **0.414**.
Pearson: mean(C) = 3.5; A centered = [-1.5, -0.5, 0.5, 1.5], C centered = [-0.5, -0.5, 0.5, 0.5].
dot = 0.75 + 0.25 + 0.25 + 0.75 = 2; |A_c| = sqrt(5) = 2.236; |C_c| = sqrt(1) = 1; Pearson = 2 / 2.236 = **0.894**.

| Pair | Euclidean similarity | Pearson |
|---|---|---|
| (A, B) | 0.333 | 1.000 |
| (A, C) | 0.414 | 0.894 |

Euclidean ranks C as A's nearest neighbour (0.414 > 0.333), but Pearson ranks B as the nearest (1.0 > 0.894). B has exactly A's taste with a lower baseline, while C has the right direction but a weaker pattern. Pearson identifies the person who agrees about which movies are better, which is what collaborative filtering needs; Euclidean is distracted by the constant offset between A and B.
<!-- ===== AI-ASSISTED END ===== -->

### Q18 - Cost of keeping similarities up to date (200M users, 20,000 products)
<!-- ===== AI-ASSISTED START (Claude): Q18 answer ===== -->
**Size.** User-user similarities: about 200M x 200M / 2 = 2 x 10^16 pairs, impossible to store or even to recompute. Item-item similarities: 20,000 x 20,000 / 2 = 2 x 10^8 pairs, which fits in memory/disk of a normal server cluster.

**Stability.** Every new rating changes the vector of one user, so user-user similarities are constantly stale (users rate sporadically, new users join all the time, and most users have very few ratings so their vectors are noisy). Item vectors have thousands or millions of ratings each, so one new rating barely changes an item's similarity; item-item similarities can be recomputed offline in batch (e.g. daily) and stay valid in between.

**Serving.** With item-based CF, recommending for a user is a lookup: take the items they rated, fetch the precomputed neighbours of those items, and aggregate. With user-based CF you must search 200M users at request time.

So the real reason item-based CF dominates in industry is scalability and stability: the item side is small and slow-changing, while the user side is huge and changes constantly, not that item-based is more accurate.
<!-- ===== AI-ASSISTED END ===== -->

## Bonus - Top-10 recommendations for one user
Predicted ratings for all items the user has not rated, using the user-based method, then a manual genre check.

In [ ]:
# ===== AI-ASSISTED START (Claude): top-N recommendations =====
def top_n_recommendations(matrix, user, movies_df, k=50, n=10, similarity_fn=pearson_similarity, min_neighbors=3):
    # 1) k most similar users (co-rated >= 2)
    sims = []
    for other in matrix.index:
        if other == user:
            continue
        v1, v2 = get_co_rated(matrix, user, other, "index")
        if len(v1) < 2:
            continue
        sims.append((other, similarity_fn(v1, v2)))
    sims.sort(key=lambda x: x[1], reverse=True)
    sims = sims[:k]
    ids = [s[0] for s in sims]
    w = np.array([s[1] for s in sims])

    # 2) similarity-weighted average per item, using only neighbours who rated that item (Section 3.5)
    R = matrix.loc[ids]
    mask = R.notna().to_numpy()
    vals = R.fillna(0).to_numpy()            # the zeros are multiplied by mask, they never count as ratings
    num = (w[:, None] * vals * mask).sum(axis=0)
    den = (np.abs(w)[:, None] * mask).sum(axis=0)
    cnt = mask.sum(axis=0)
    pred = np.where((cnt >= min_neighbors) & (den > 0), num / np.where(den == 0, 1, den), np.nan)
    pred = pd.Series(pred, index=matrix.columns)

    # 3) only items the user has not rated
    pred = pred[matrix.loc[user].isna()].dropna().sort_values(ascending=False).head(n)
    out = pred.rename("predicted_rating").reset_index().merge(movies_df, on="movieId")
    return out
# ===== AI-ASSISTED END =====

user = 1   # TODO: choose your own user
recs = top_n_recommendations(matrix, user, movies)
display(recs[["title", "genres", "predicted_rating"]])

# Manual sanity check: what does this user actually like?
hist = ratings[ratings.userId == user].merge(movies, on="movieId").sort_values("rating", ascending=False)
display(hist.head(10)[["title", "genres", "rating"]])
print(hist[hist.rating >= 4].genres.str.split("|").explode().value_counts().head(5))
print(recs.genres.str.split("|").explode().value_counts().head(5))

**TODO (own words):** compare the two genre counts above: do the recommended genres match the genres the user rated 4+ ?